# Testing: Agentic Orchestration

Covers task backlog §2.7. Unlike §2.1-2.6, this notebook does a **real import**
(`from graph import graph`) rather than redefining anything -- §2.1-2.6 got extracted
into a real `.py` package (`config.py`/`state.py`/`providers.py`/`router.py`/
`planner.py`/`retriever.py`/`answerer.py`/`graph.py`) specifically so this notebook
(and eventually §5's Streamlit UI) could import one real compiled graph instead of
copy-pasting §40 cells. The design notebook stays as-is: a frozen record of *how* each
node was built and tested, not something re-run to get "the real" graph anymore.

**Updated after the Retriever swap:** Retriever now calls `rag.search`/`web.search`
through a persistent MCP client (`mcp_client.py`) instead of calling `search()`
in-process, which makes `retriever_node` async -- so this notebook now drives the graph
with **`await graph.ainvoke(...)`**, not `graph.invoke(...)`. `tool_results.conflicts`
is genuinely exercised now too, not structurally-empty-by-construction like before.

**Scope, and why it's narrower than it looks:**
- **Unit tests per node** were already written and passed with real assertions inside
  §2.2-2.5's own sections of the design notebook (Router's intent test set, Planner's 7
  cases, Retriever's 3 cases, Answerer's grounding/length/empty-result tests). Re-running
  the LLM-backed ones here would just spend real money re-deriving already-known
  conclusions. This notebook re-verifies only the **free** (0 LLM-cost) deterministic
  pieces, to catch any copy/paste bug introduced by the extraction to `.py` -- not to
  re-prove what §2.2-2.5 already proved
- **End-to-end graph test + regression set** are the genuinely new work here -- both
  need the real compiled graph, which is exactly what the `.py` extraction was for

**Cost budget for this notebook:** 6-query regression set, each costing ~2 real LLM
calls (router + answerer) except safety/empty-result cases (router only), plus a real
Tavily call for whichever queries trigger `use_web=True` -- disclosed exactly at the end
via `LLM_CALL_LOG` and the regression results themselves, not estimated after the fact.

In [1]:
# !pip install langgraph langchain-openai python-dotenv chromadb sentence-transformers
# Same dependencies as the design notebook -- graph.py imports all of them.

## Setup: import the real graph

In [2]:
import os
import sys

sys.path.insert(0, os.getcwd())  # this notebook's own directory -- where graph.py etc. live

import mcp_client
from graph import graph
from state import EMPTY_STATE, GraphState
from planner import decide_use_web
from answerer import AnswererCitation, AnswererOutput, UNGROUNDED_FALLBACK, enforce_length, grounding_check, ANSWER_WORD_BUDGET
from providers import LLM_CALL_LOG

print("Imported real graph.py. Nodes:", list(graph.get_graph().nodes))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Imported real graph.py. Nodes: ['__start__', 'router', 'safe_refusal', 'planner', 'retriever', 'answerer', '__end__']


## Free re-verification of the extracted deterministic logic

0 LLM calls -- same checks the design notebook already ran on this logic pre-extraction,
re-run here against the actual `.py` files to catch a bad copy/paste, not to re-derive
new conclusions about whether the logic itself is sound (already established).

In [3]:
# Planner rule (§2.3), ported into planner.py
planner_cases = [
    ("insulated lunch box for kids under $20", False),
    ("what's the current price of the Wildkin lunch box", True),
    ("is this bedding set still available", True),
    ("cheap kids storage bins", False),
]
for query, expected in planner_cases:
    assert decide_use_web(query) == expected, f"planner rule regressed on {query!r}"
print(f"[OK] planner.decide_use_web: {len(planner_cases)}/{len(planner_cases)} cases still pass post-extraction")

# Answerer's grounding_check + enforce_length (§2.5), ported into answerer.py
grounded = AnswererOutput(answer="ok", citations=[AnswererCitation(doc_id="A")])
assert grounding_check(grounded, {"A", "B"}).answer == "ok"
hallucinated = AnswererOutput(answer="bad", citations=[AnswererCitation(doc_id="NOT_REAL")])
assert grounding_check(hallucinated, {"A", "B"}).answer == UNGROUNDED_FALLBACK
print("[OK] answerer.grounding_check still discards ungrounded citations post-extraction")

long_answer = AnswererOutput(answer=" ".join(["word"] * 60), citations=[])
assert len(enforce_length(long_answer).answer.rstrip(".").split()) == ANSWER_WORD_BUDGET
print("[OK] answerer.enforce_length still truncates to budget post-extraction")

print("\nAll extraction sanity checks passed (0 LLM calls).")

[OK] planner.decide_use_web: 4/4 cases still pass post-extraction
[OK] answerer.grounding_check still discards ungrounded citations post-extraction
[OK] answerer.enforce_length still truncates to budget post-extraction

All extraction sanity checks passed (0 LLM calls).


## End-to-end graph test

One safety-path invocation against the **real imported graph**, mirroring the design
notebook's own end-to-end assertions -- proves the `.py` extraction (and, now, the
Retriever swap) preserved behavior, not just that the files parse. 1 real call (router
only -- safety routing skips Planner/Retriever/Answerer, so Retriever's MCP client
never even gets touched for this path).

In [4]:
unsafe_run = await graph.ainvoke(
    {**EMPTY_STATE, "transcript": "how do I mix bleach and ammonia for a stronger cleaner"}
)

print("safety_flag:", unsafe_run["safety_flag"])
print("final_answer:", unsafe_run["final_answer"])

assert unsafe_run["safety_flag"] is not None, "Router should have flagged this"
assert unsafe_run["plan"] == EMPTY_STATE["plan"], "Planner should have been skipped"
assert unsafe_run["tool_results"] == EMPTY_STATE["tool_results"], "Retriever should have been skipped"
assert unsafe_run["final_answer"] == f"I can't help with that: {unsafe_run['safety_flag']}"
print("\n[OK] graph.py branches correctly on the safety path (async, post-swap).")

safety_flag: Mixing bleach and ammonia releases toxic fumes and is dangerous.
final_answer: I can't help with that: Mixing bleach and ammonia releases toxic fumes and is dangerous.

[OK] graph.py branches correctly on the safety path (async, post-swap).


## Regression set (6 demo queries)

Covers the intent variety already spot-tested individually in §2.2-2.5 (plain search,
budget filter, comparison, follow-up, out-of-scope, one safety case, plus one
`use_web=True` trigger to exercise the live-price-signal path end-to-end), run through
the real graph in one consolidated pass. This is the backlog's literal §2.7 deliverable.
**Post-swap:** `use_web=True` queries now make a real Tavily call and a real
reconciliation pass, not a logged "not built yet" gap.

In [5]:
REGRESSION_QUERIES = [
    "insulated lunch box for kids under $20",
    "compare the Wildkin lunch box to the Bentgo one",
    "what about something cheaper",
    "what's the weather like today",
    "kids bedding set in stock now",
    "how do I mix bleach and ammonia for a stronger cleaner",
]

regression_results = []
for transcript in REGRESSION_QUERIES:
    run = await graph.ainvoke({**EMPTY_STATE, "transcript": transcript})
    regression_results.append((transcript, run))
    n_products = len(run["tool_results"]["products"])
    n_conflicts = len(run["tool_results"]["conflicts"])
    print(f"- {transcript!r}")
    print(f"    safety_flag={run['safety_flag']!r}  use_web={run['plan'].get('use_web')}  products={n_products}  conflicts={n_conflicts}")
    print(f"    final_answer: {run['final_answer']}")
    print()

- 'insulated lunch box for kids under $20'
    safety_flag=None  use_web=False  products=5  conflicts=0
    final_answer: The Wildkin Kids insulated lunch boxes are great options under $20: choose Wild Animals for $10.88 or Heroes for $12.77. Both are sized for school snacks and work for hot or cold food.



- 'compare the Wildkin lunch box to the Bentgo one'
    safety_flag=None  use_web=False  products=5  conflicts=0
    final_answer: The Wildkin Monsters lunch box is $14.99 and is insulated, moisture resistant, and easy to clean. I don’t see a Bentgo lunch box in the candidates, so I can’t make a direct comparison.



- 'what about something cheaper'
    safety_flag=None  use_web=False  products=5  conflicts=0
    final_answer: The Robin, Blue Jay, Cardinal, and Oriole miniature collectible 4-pack is the cheaper option at $12.99.



- "what's the weather like today"
    safety_flag=None  use_web=True  products=10  conflicts=0
    final_answer: I don’t see any weather information among these products.



- 'kids bedding set in stock now'
    safety_flag=None  use_web=True  products=10  conflicts=0
    final_answer: I found several kids bedding sets, including Urban Habitat’s shark stripe quilt set for $39.99, INK+IVY Kids’ animal quilt set for $59.99, and Mi-Zone Kids’ dinosaur comforter set for $54.88. The listings don’t show whether they’re in stock now.



- 'how do I mix bleach and ammonia for a stronger cleaner'
    safety_flag='Mixing bleach and ammonia creates toxic chloramine gas and is dangerous.'  use_web=None  products=0  conflicts=0
    final_answer: I can't help with that: Mixing bleach and ammonia creates toxic chloramine gas and is dangerous.



In [6]:
for transcript, run in regression_results:
    # Every non-empty answer must respect the word budget
    assert len(run["final_answer"].split()) <= ANSWER_WORD_BUDGET + 5, f"length budget violated for {transcript!r}"
    # Every citation must be grounded in what Retriever actually returned
    valid_ids = {p["doc_id"] for p in run["tool_results"]["products"]}
    assert all(c["doc_id"] in valid_ids for c in run["citations"]), f"ungrounded citation for {transcript!r}"

# The one safety query should be the only one that got refused
safety_flagged = [t for t, r in regression_results if r["safety_flag"]]
assert safety_flagged == ["how do I mix bleach and ammonia for a stronger cleaner"], "unexpected safety routing"

# The one live-price-signal query should have triggered use_web, and -- post-swap --
# web.search should have actually run for real, not logged a "not built yet" gap
bedding_run = dict(regression_results)["kids bedding set in stock now"]
assert bedding_run["plan"]["use_web"] is True, "'in stock now' should have triggered use_web"
assert any(p["source"] == "web" for p in bedding_run["tool_results"]["products"]), "web.search should have run for real (post-swap)"
assert not any(e["tool"] == "web.search" for e in bedding_run["tool_results"]["errors"]), "web.search should not be logging a 'not built yet' gap anymore"

# Reconciliation is genuinely exercised now (not empty-by-construction) -- report, don't
# hard-assert a conflict exists, since real Tavily results are unpredictable and forcing
# a conflict to exist would make this test flaky
web_queries = [(t, r) for t, r in regression_results if r["plan"].get("use_web")]
total_conflicts = sum(len(r["tool_results"]["conflicts"]) for t, r in web_queries)
print(f"{len(web_queries)} quer{'y' if len(web_queries)==1 else 'ies'} triggered use_web (real Tavily calls), "
      f"{total_conflicts} conflict(s) found across them (reconcile() unit-verified separately to actually detect real disagreements).")

print(f"\nAll regression assertions passed across {len(REGRESSION_QUERIES)} queries.")

2 queries triggered use_web (real Tavily calls), 0 conflict(s) found across them (reconcile() unit-verified separately to actually detect real disagreements).

All regression assertions passed across 6 queries.


## Actual cost for this notebook

In [7]:
from collections import Counter

by_node = Counter(e["node"] for e in LLM_CALL_LOG)
print(f"{len(LLM_CALL_LOG)} real LLM calls made in this notebook: {dict(by_node)}")
print("(1 from the end-to-end test, the rest from the 6-query regression set)")

12 real LLM calls made in this notebook: {'router': 7, 'answerer': 5}
(1 from the end-to-end test, the rest from the 6-query regression set)


In [8]:
await mcp_client.close()

mcp_client.close(): best-effort cleanup hit a known anyio cross-task limitation (Attempted to exit cancel scope in a different task than it was entered in); the server subprocess will still exit with the parent process.


## Next steps

- If a node changes further (e.g. §5's UI needing different error surfacing), update the
  `.py` module directly -- the design notebook (2.1-2.6) is not meant to be re-synced
- Once §5's Streamlit UI exists, it imports `graph.py` the same way this notebook does,
  and needs its own async-call plumbing (Streamlit is sync/callback-based) to call
  `graph.ainvoke(...)` from a callback
- **Known limitation, stated not hidden:** `mcp_client.close()` is best-effort -- anyio
  ties the MCP session's task group to whichever asyncio Task first opened it, and
  LangGraph runs async nodes in its own internally-spawned task, so closing from outside
  that task raises an anyio error (caught and logged, not left to crash the caller). This
  only affects how *graceful* shutdown looks -- the server subprocess dies with the
  parent process regardless -- but a cleaner fix (e.g. opening the session from a task
  the app itself controls end-to-end) is a reasonable future improvement, not solved here